In [54]:
import pandas as pd
import spacy

In [55]:
df = pd.read_csv("./enron_spam_data.csv")

In [56]:
display(df)

,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14
...,...,...,...,...,...
33711,33711,= ? iso - 8859 - 1 ? q ? good _ news _ c = eda...,"hello , welcome to gigapharm onlinne shop .\np...",spam,2005-07-29
33712,33712,all prescript medicines are on special . to be...,i got it earlier than expected and it was wrap...,spam,2005-07-29
33713,33713,the next generation online pharmacy .,are you ready to rock on ? let the man in you ...,spam,2005-07-30
33714,33714,bloow in 5 - 10 times the time,learn how to last 5 - 10 times longer in\nbed ...,spam,2005-07-30


In [57]:
df['Spam/Ham'].value_counts()

Spam/Ham
spam    17171
ham     16545
Name: count, dtype: int64

In [58]:
df['Message'].fillna(' ')

0                                                         
1        gary , production from the high island larger ...
2                   - calpine daily gas nomination 1 . doc
3        fyi - see note below - already done .\nstella\...
4        fyi .\n- - - - - - - - - - - - - - - - - - - -...
                               ...                        
33711    hello , welcome to gigapharm onlinne shop .\np...
33712    i got it earlier than expected and it was wrap...
33713    are you ready to rock on ? let the man in you ...
33714    learn how to last 5 - 10 times longer in\nbed ...
33715    hi : )\ndo you need some softwares ? i can giv...
Name: Message, Length: 33716, dtype: object

In [59]:
df['combined'] = df['Subject'].fillna(' ') + df['Message'].fillna(' ')

In [60]:
len(' '.join(df['combined']))

50773497

In [61]:
# too big! need to sample...

# first randomize the order
df_sample = df.sample(frac=0.01, ignore_index=True).copy()


In [62]:
len(' '.join(df_sample['combined']))

418251

## Spacy

In [63]:
from collections import Counter

nlp = spacy.load("en_core_web_sm")
nlp.max_length = 1*10**6

text = ' '.join(df_sample['combined'])
doc = nlp(text)

tokens = [token.text for token in doc]
print("Total tokens:", len(tokens))
print("Unique tokens:", len(set(tokens)))

word_counts = Counter(tokens)
print("Most common words:", word_counts.most_common(10))

noun_counts = Counter(token.text for token in doc if token.pos_ == "NOUN")
print("Most common nouns:", noun_counts.most_common(10))


Total tokens: 97922
Unique tokens: 12375
Most common words: [('\n', 7003), ('.', 5925), ('-', 4312), (',', 2827), ('/', 2063), ('the', 1982), (':', 1876), ('to', 1576), ('and', 1113), ('of', 1033)]
Most common nouns: [('com', 214), ('enron', 213), ('|', 184), ('http', 176), ('subject', 120), ('=', 118), ('pm', 103), ('company', 98), ('email', 93), ('gas', 92)]


## Gensim

In [48]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/amarks-b/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [49]:
from gensim import corpora
from gensim.models import LdaModel
from nltk.tokenize import word_tokenize

  
documents = df_sample['combined'].values
tokenized_docs = [word_tokenize(doc.lower()) for doc in documents]
dictionary = corpora.Dictionary(tokenized_docs)
corpus = [dictionary.doc2bow(tokens) for tokens in tokenized_docs]
lda_model = LdaModel(corpus, num_topics=2, id2word=dictionary, passes=10)
for topic in lda_model.print_topics():
    print(topic)

(0, '0.077*"-" + 0.055*"." + 0.031*":" + 0.025*"/" + 0.023*"," + 0.018*">" + 0.014*"the" + 0.012*"?" + 0.012*"to" + 0.009*"\'"')
(1, '0.044*"." + 0.039*"," + 0.032*"the" + 0.023*"to" + 0.018*"and" + 0.016*"of" + 0.012*"-" + 0.012*"a" + 0.011*"in" + 0.010*"_"')


## word2vec (in gensim)

In [68]:
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import nltk

# Download punkt tokenizer if you haven't already
try:
    nltk.data.find('tokenizers/punkt')
except nltk.downloader.DownloadError:
    nltk.download('punkt')

# Your training data (list of sentences/documents)
data = df_sample['combined'].values

# 1. Tokenize the data (split sentences into words)
tokenized_data = [word_tokenize(sentence.lower()) for sentence in data]

# 2. Train the Word2Vec model
model = Word2Vec(sentences=tokenized_data, vector_size=100, window=5, min_count=1, workers=4)
# - sentences: The tokenized training data.
# - vector_size: Dimensionality of the word vectors (e.g., 100 features per word).
# - window: Maximum distance between the current and predicted word within a sentence.
# - min_count: Ignores all words with total frequency lower than this.
# - workers: Number of worker threads to train the model (for faster training).

# 3. Access word vectors (example)
word = "meeting"
if word in model.wv:
    vector = model.wv[word]
    print(f"Vector for '{word}': {vector}")
else:
    print(f"Word '{word}' not found in the vocabulary.")

# 4. Find similar words (example)
similar_words = model.wv.most_similar("enron", topn=3)
print(f"\nWords similar to 'enron': {similar_words}")


word = "last"
similar_words = model.wv.most_similar(f"{word}", topn=3)
print(f"\nWords similar to '{word}': {similar_words}")



Vector for 'meeting': [ 0.07598456  0.25815853  0.15329161  0.23427546 -0.24182034 -0.3388962
  0.11956709  0.66435874 -0.09583826 -0.06792221 -0.15688437 -0.2122039
  0.06496997  0.1228454  -0.06722878 -0.20171806  0.328949   -0.24359274
  0.10923555 -0.694488    0.1023971  -0.1412128   0.11255617 -0.14989693
 -0.05843696  0.09250896 -0.30178192  0.08379035 -0.08410117 -0.01401008
  0.6251305   0.13659641  0.29666463 -0.41174257 -0.21837854 -0.10442437
 -0.03088508 -0.22954705 -0.37569633 -0.63194203 -0.18419434 -0.32331872
 -0.03012101  0.1053271   0.08314783 -0.2839787  -0.16755998 -0.2006064
  0.00843723  0.20746368  0.12767613  0.10929864 -0.19258071  0.14625128
 -0.16612007 -0.11285333  0.18569511 -0.18526267 -0.3805113   0.00112486
  0.24471447  0.17426245  0.0228304   0.2915774  -0.541877    0.40055716
  0.15532976  0.18609554 -0.43380988  0.3747296  -0.15429622  0.41897312
  0.26666224 -0.17206271  0.43318158 -0.14772995  0.31100678  0.30957997
 -0.07040372  0.35078752 -0.3387